# Binary ARCiS Retrieval

This notebook shows the minimal setup for a binary retrieval with a generic multi-component simulator wrapper. FlopPITy samples one parameter vector containing both components. The wrapper splits that vector, runs ARCiS once per component, and sums the returned spectra.

The example assumes both components use the same ARCiS input file and therefore the same fitted-parameter list. Component-specific parameters get `_1` and `_2` suffixes. Shared parameters are sampled once and reused for both components. The example also includes an optional sampled column fraction for weighted combinations.

In [ ]:
from floppity import Retrieval
from floppity.simulators import ARCiS, make_binary_simulator, read_ARCiS_input

## Paths

Edit these paths for your local ARCiS setup.

In [ ]:
arcis_input = "path/to/arcis_input.in"
arcis_executable = "/usr/local/bin/ARCiS"

arcis_output_dir = "output_ARCiS_multi_component"
retrieval_output_dir = "output_FlopPITy_binary"

## Read The Single-Component ARCiS Setup

`read_ARCiS_input` returns the fitted parameters and observation files from one ARCiS input file. For a binary retrieval, the helper below duplicates component-specific parameters and keeps selected parameters shared.

In [ ]:
single_component_pars, obs_dict = read_ARCiS_input(arcis_input)

# Edit this list to choose parameters that should be identical in both components.
# Names must match the single-component ARCiS fitpar keywords.
shared_parameters = [
    # "log_h2o",
    # "log_ch4",
]

# Set this to None for a direct sum, or use a sampled fraction for weighted columns.
weight_parameters = {"column_fraction": (0, 1)}
# weight_parameters = None

binary_simulator, binary_parameters = make_binary_simulator(
    ARCiS,
    single_component_pars,
    shared_parameters=shared_parameters,
    weight_parameters=weight_parameters,
)

print(f"Single-component parameters: {len(single_component_pars)}")
print(f"Binary retrieval parameters: {len(binary_parameters)}")
print(list(binary_parameters)[:5])

## Build The Retrieval

The retrieval uses the wrapped simulator returned by `make_binary_simulator`.

In [ ]:
R = Retrieval(binary_simulator, obs_type="emis")
R.parameters = binary_parameters
R.get_obs(obs_dict)

# Optional preprocessing. Keep this disabled until you are sure your spectra
# are strictly positive after ARCiS and FlopPITy's emission clipping.
# R.preprocessing = ["log"]

## Runtime Options

Start small while testing that ARCiS runs and the parameter ordering is correct. Increase `n_samples`, `n_samples_init`, and `n_rounds` for a real retrieval.

In [ ]:
ARCiS_kwargs = {
    "input_file": arcis_input,
    "output_dir": arcis_output_dir,
    "ARCiS_dir": arcis_executable,
    "num_threads": "4",
    "save_atmosphere": True,
}

flow_kwargs = {
    "flow": "nsf",
    "bins": 4,
    "transforms": 10,
    "blocks": 3,
    "hidden": 64,
    "dropout": 0.1,
}

training_kwargs = {
    "stop_after_epochs": 10,
    "num_atoms": 20,
    "learning_rate": 1e-3,
}

## Run

With `weight_parameters={"column_fraction": (0, 1)}`, the wrapper combines spectra as `column_fraction * component_1 + (1 - column_fraction) * component_2`. With `weight_parameters=None`, it directly sums the two component spectra.

In [ ]:
R.run(
    flow_kwargs=flow_kwargs,
    training_kwargs=training_kwargs,
    simulator_kwargs=ARCiS_kwargs,
    output_dir=retrieval_output_dir,
    n_threads=2,
    n_rounds=2,
    n_samples_init=16,
    n_samples=16,
    save_data=True,
    sample_prior_method="sobol",
)

## Inspect The Result

After the run, the posterior contains the full binary parameter vector. The first component's parameters have `_1` suffixes and the second component's parameters have `_2` suffixes.

In [ ]:
fig = R.plot_corner(n_samples=2000)